<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Developing Applications with Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">3. DataFrame Relational Operations</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 3.1 Lecture - DataFrame Relational Operations
### Duration - 27 Minutes

> *Combining distributed datasets with joins and set operations*

This lecture covers the relational side of the DataFrame API: join types and conditions, set operations for comparing datasets, and the performance practices that keep joins fast on distributed data.

## Learning Objectives

**By the end of this lecture, you will be able to:**

- Describe different types of DataFrame joins (inner, outer, left, right, cross) and their applications in combining datasets
- Use set operations like union, intersect, and subtract for combining or finding differences between DataFrames
- Apply best practices for optimizing relational operations, including choosing appropriate join strategies and handling data skew

## A. DataFrame Join Operations

Joins combine two DataFrames based on matching key column(s):

- Multiple join types supported: inner, outer, left, right, cross
- The join condition can use single or multiple columns
- Column naming conflicts are handled with DataFrame aliases or by renaming columns before the join

| Join Type (`how=`) | Returns |
|--------------------|---------|
| __`inner`__ (default) | Rows with matching keys in both DataFrames |
| __`left`__ | All left rows, with nulls where the right side has no match |
| __`right`__ | All right rows, with nulls where the left side has no match |
| __`full_outer`__ | All rows from both sides, with nulls for missing matches |
| __`left_semi`__ | Left rows that have a match (no right columns) |
| __`left_anti`__ | Left rows that have no match |
| __`cross`__ | Every combination of rows (use with care) |

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Anatomy of a distributed join</strong>
      </div>
    </div>
  </summary>
  <div style="width:100%;max-width:820px;margin:14px auto"><svg viewBox="0 0 1120 360" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Anatomy of a join" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block"><title>Anatomy of a join</title><defs><marker id="ah_asp2_join" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,1 L10,5 L0,9 z" fill="#3C4B52"/></marker></defs><rect x="40" y="40" width="300" height="110" rx="10" fill="#DCE0E2" stroke="#618794" stroke-width="2"/><text x="190.0" y="100.9" text-anchor="middle" font-size="21" fill="#0B2026" font-weight="700">users DataFrame</text><rect x="40" y="210" width="300" height="110" rx="10" fill="#DCE0E2" stroke="#618794" stroke-width="2"/><text x="190.0" y="270.9" text-anchor="middle" font-size="21" fill="#0B2026" font-weight="700">orders DataFrame</text><polygon points="512,120 648,120 690,180.0 648,240 512,240 470,180.0" fill="#FFF3E0" stroke="#FF9800" stroke-width="2"/><text x="580.0" y="187.0" text-anchor="middle" font-size="20" font-weight="700" fill="#0B2026">join on user_id</text><rect x="800" y="120" width="290" height="120" rx="10" fill="#E8F5E9" stroke="#4CAF50" stroke-width="2"/><text x="945.0" y="175.7" text-anchor="middle" font-size="21" fill="#0B2026" font-weight="700">Combined DataFrame</text><text x="945.0" y="199.4" text-anchor="middle" font-size="16" fill="#4A5A62" font-style="italic">user and order columns</text><path d="M340,95 L476,154.0" fill="none" stroke="#3C4B52" stroke-width="2.2" marker-end="url(#ah_asp2_join)"/><path d="M340,265 L476,206.0" fill="none" stroke="#3C4B52" stroke-width="2.2" marker-end="url(#ah_asp2_join)"/><path d="M690,180.0 L800,180.0" fill="none" stroke="#3C4B52" stroke-width="2.2" marker-end="url(#ah_asp2_join)"/></svg></div>
</details>

### join Examples

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #999"># Basic join example</span>
<span style="color: #CCC">df1</span><span style="color: #67CDCC">.</span><span style="color: #CCC">join(df2,</span> <span style="color: #7EC699">"user_id"</span><span style="color: #CCC">)</span>  <span style="color: #999"># Inner join by default</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #999"># Specifying join type</span>
<span style="color: #CCC">df1</span><span style="color: #67CDCC">.</span><span style="color: #CCC">join(df2,</span>
    <span style="color: #CCC">on</span><span style="color: #67CDCC">=</span><span style="color: #CCC">df1</span><span style="color: #67CDCC">.</span><span style="color: #CCC">user_id</span> <span style="color: #67CDCC">==</span> <span style="color: #CCC">df2</span><span style="color: #67CDCC">.</span><span style="color: #CCC">user_id,</span>
    <span style="color: #CCC">how</span><span style="color: #67CDCC">=</span><span style="color: #7EC699">"left"</span><span style="color: #CCC">)</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #999"># Multiple conditions with DataFrame aliases</span>
<span style="color: #CC99CD">from</span><span style="color: #CCC"> pyspark.sql.functions </span><span style="color: #CC99CD">import</span> <span style="color: #CCC">col</span>
<span style="color: #CCC">users</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"u"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">join(</span>
    <span style="color: #CCC">orders</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"o"</span><span style="color: #CCC">),</span>
    <span style="color: #CCC">[col(</span><span style="color: #7EC699">"u.id"</span><span style="color: #CCC">)</span> <span style="color: #67CDCC">==</span> <span style="color: #CCC">col(</span><span style="color: #7EC699">"o.user_id"</span><span style="color: #CCC">),</span>
     <span style="color: #CCC">col(</span><span style="color: #7EC699">"u.region"</span><span style="color: #CCC">)</span> <span style="color: #67CDCC">==</span> <span style="color: #CCC">col(</span><span style="color: #7EC699">"o.region"</span><span style="color: #CCC">)],</span>
    <span style="color: #7EC699">"inner"</span><span style="color: #CCC">)</span></code></pre>



## B. Set Operations

Set operations treat DataFrames as sets of rows, helping you understand relationships within datasets. Given `A = [1, 2, 3]` and `B = [2, 3, 4]`:

- **UNION** - includes all unique elements from both sets, no duplicates: `A UNION B -> [1, 2, 3, 4]`
- **INTERSECTION** - includes elements in both sets: `A INTERSECT B -> [2, 3]`
- **SUBTRACTION** - includes elements in A only: `A MINUS B -> [1]`

All DataFrame set operations require matching schemas. The methods:

| Operation | Description | Example |
|-----------|-------------|---------|
| __`union()`__ | Combines all rows from both DataFrames by position | `df1.union(df2)` |
| __`unionByName()`__ | Combines DataFrames matching by column names | `df1.unionByName(df2)` |
| __`intersect()`__ | Returns only rows present in both DataFrames | `df1.intersect(df2)` |
| __`subtract()`__ | Returns rows in the first DataFrame but not in the second | `df1.subtract(df2)` |

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#8505;&#65039;</span>
<div>
<strong style="color: #0d47a1; font-size: 1.1em">Duplicates</strong>
<p style="margin: 8px 0 0 0; color: #333"><code>union()</code> in Spark behaves like SQL <code>UNION ALL</code> - it keeps duplicates (deduplicate with <code>distinct()</code>). <code>intersect()</code> and <code>subtract()</code> remove duplicates; use the <code>intersectAll()</code> and <code>exceptAll()</code> variants to preserve them.</p>
</div>
</div>
</div>

## C. Join Performance Considerations

<p style="font-size: 1em; line-height: 1.6; color: #333">Optimizing relational operations in a distributed environment:</p>

- **Join strategy selection** - Spark automatically chooses the join strategy; reference the smaller DataFrame first where practical
- **Broadcast joins** - Spark can avoid shuffling the large side entirely by copying a small DataFrame to every Executor; use the `broadcast()` hint to request this explicitly

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #999"># Good - small_df is smaller</span>
<span style="color: #CCC">small_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">join(large_df,</span> <span style="color: #7EC699">"key"</span><span style="color: #CCC">)</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #999"># Less efficient - large_df is larger</span>
<span style="color: #CCC">large_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">join(small_df,</span> <span style="color: #7EC699">"key"</span><span style="color: #CCC">)</span>
<span style="color: #67CDCC">&lt;</span><span style="color: #CCC">br</span><span style="color: #67CDCC">/&gt;</span>
<span style="color: #999"># Broadcasting a small DataFrame</span>
<span style="color: #CC99CD">from</span><span style="color: #CCC"> pyspark.sql.functions </span><span style="color: #CC99CD">import</span> <span style="color: #CCC">broadcast</span>
<span style="color: #CCC">large_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">join(broadcast(small_df),</span> <span style="color: #7EC699">"key"</span><span style="color: #CCC">)</span></code></pre>



### More Join Performance Considerations

- **Data skew handling**
  - Uneven distribution of join keys can impact performance
  - Consider repartitioning in some cases
- **Memory management**
  - Monitor shuffle spill metrics for joins
  - Consider caching frequently joined DataFrames
  - Use projection to select only needed columns before joining

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#128161;</span>
<div>
<strong style="color: #00695c; font-size: 1.1em">Use the Spark UI</strong>
<p style="margin: 8px 0 0 0; color: #333">Use the Spark UI to monitor join performance and identify bottlenecks - the SQL tab shows the chosen join strategy (e.g. <code>BroadcastHashJoin</code> vs <code>SortMergeJoin</code>) and per-stage shuffle metrics.</p>
</div>
</div>
</div>

## Summary

- Joins combine DataFrames on key columns; choose the join type (`inner`, `left`, `right`, `full_outer`, `semi`, `anti`, `cross`) based on which unmatched rows you need
- Set operations (`union()`, `unionByName()`, `intersect()`, `subtract()`) compare whole rows and require matching schemas
- Broadcast small DataFrames to avoid shuffling large ones; project columns before joining; watch for data skew
- The Spark UI shows which join strategy the optimizer picked and where the time went

## Knowledge Check

Answer the questions below to check your understanding of this lecture.

<div id="mcqRoot"></div>

<script>
(function() {
    var questions = [
        {
                "question": "Which set operation should you use when the two DataFrames have the same columns but in a different order?",
                "options": [
                        {
                                "id": "a",
                                "text": "union()"
                        },
                        {
                                "id": "b",
                                "text": "unionByName()"
                        },
                        {
                                "id": "c",
                                "text": "intersect()"
                        },
                        {
                                "id": "d",
                                "text": "subtract()"
                        }
                ],
                "correct": "b",
                "explanation": "union() matches columns by position, so differing column orders silently combine the wrong columns. unionByName() matches columns by name regardless of order."
        },
        {
                "question": "When performing a join operation in Spark, which statement is correct?",
                "options": [
                        {
                                "id": "a",
                                "text": "Cross joins are the default join type"
                        },
                        {
                                "id": "b",
                                "text": "Join conditions can only use a single column"
                        },
                        {
                                "id": "c",
                                "text": "Broadcast hints can optimize joins with small tables"
                        },
                        {
                                "id": "d",
                                "text": "Joins always require identical column names on both sides"
                        }
                ],
                "correct": "c",
                "explanation": "broadcast() copies the small DataFrame to every Executor so the large side never shuffles. Inner is the default join type, conditions can use multiple columns, and differing column names are handled with join expressions."
        },
        {
                "question": "Which is NOT a recommended practice for optimizing join performance in Spark?",
                "options": [
                        {
                                "id": "a",
                                "text": "Using broadcast hints for small tables"
                        },
                        {
                                "id": "b",
                                "text": "Caching frequently joined DataFrames"
                        },
                        {
                                "id": "c",
                                "text": "Always using cross joins for better distribution"
                        },
                        {
                                "id": "d",
                                "text": "Repartitioning to handle data skew"
                        }
                ],
                "correct": "c",
                "explanation": "A cross join produces every combination of rows - its cost grows multiplicatively and it is almost never a performance optimization. The other three are standard join tuning practices."
        },
        {
                "question": "Which join type returns only the rows from the left DataFrame that have NO match in the right DataFrame?",
                "options": [
                        {
                                "id": "a",
                                "text": "left"
                        },
                        {
                                "id": "b",
                                "text": "left_semi"
                        },
                        {
                                "id": "c",
                                "text": "left_anti"
                        },
                        {
                                "id": "d",
                                "text": "full_outer"
                        }
                ],
                "correct": "c",
                "explanation": "left_anti is the 'NOT EXISTS' of joins: it keeps left rows without a match and returns only the left columns. left_semi is its opposite, keeping left rows that do have a match."
        }
];
    
    var currentIndex = 0;
    var score = 0;
    var selectedAnswer = null;
    var showingFeedback = false;
    
    function init() {
        currentIndex = 0;
        score = 0;
        selectedAnswer = null;
        showingFeedback = false;
        render();
    }
    
    function render() {
        var root = document.getElementById('mcqRoot');
        var html = '';
        var isComplete = currentIndex >= questions.length;
        
        html += '<div style="font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif; width: 100%; margin: 10px 0; padding: 24px; background: #f5f7fa; border-radius: 12px; border: 1px solid #e0e0e0; box-sizing: border-box;">';
        
        // Header
        html += '<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">';
        html += '<div>';
        html += '<div style="font-size: 1.4em; font-weight: 600; color: #333;">📝 Multiple Choice Quiz</div>';
        html += '<div style="font-size: 0.95em; color: #666; margin-top: 4px;">Knowledge Check</div>';
        html += '</div>';
        html += '<div style="display: flex; gap: 12px; align-items: center;">';
        html += '<span style="padding: 8px 16px; border-radius: 20px; font-weight: 600; font-size: 0.9em; background: #e8f5e9; color: #2e7d32;">Score: ' + score + '/' + questions.length + '</span>';
        html += '<span style="padding: 8px 16px; border-radius: 20px; font-weight: 600; font-size: 0.9em; background: #e3f2fd; color: #1976d2;">Q' + (currentIndex + 1) + '/' + questions.length + '</span>';
        html += '</div></div>';
        
        if (isComplete) {
            // Results
            var percentage = Math.round(score / questions.length * 100);
            html += '<div style="text-align: center; padding: 40px 20px; background: #fff; border-radius: 12px; border: 2px solid #e0e0e0;">';
            html += '<div style="font-size: 3em; margin-bottom: 16px;">' + (percentage >= 75 ? '🏆' : percentage >= 50 ? '👍' : '📚') + '</div>';
            html += '<div style="font-size: 1.8em; font-weight: 700; color: #333; margin-bottom: 8px;">Quiz Complete!</div>';
            html += '<div style="font-size: 1.2em; color: #666; margin-bottom: 24px;">You scored ' + score + ' out of ' + questions.length + ' (' + percentage + '%)</div>';
            html += '<button id="retakeBtn" style="padding: 14px 32px; background: #1976d2; color: white; border: none; border-radius: 8px; cursor: pointer; font-size: 1.1em; font-weight: 600;">Retake Quiz</button>';
            html += '</div>';
        } else {
            var q = questions[currentIndex];
            
            // Question
            html += '<div style="background: #fff; border-radius: 12px; border: 2px solid #e0e0e0; padding: 24px; margin-bottom: 20px;">';
            html += '<div style="font-size: 1.15em; color: #333; line-height: 1.6;">' + q.question + '</div>';
            html += '</div>';
            
            // Options
            html += '<div style="display: flex; flex-direction: column; gap: 12px; margin-bottom: 20px;">';
            
            q.options.forEach(function(opt) {
                var isSelected = selectedAnswer === opt.id;
                var isCorrect = opt.id === q.correct;
                var bgColor = '#fff';
                var borderColor = '#e0e0e0';
                var textColor = '#333';
                
                if (showingFeedback) {
                    if (isCorrect) {
                        bgColor = '#e8f5e9';
                        borderColor = '#4caf50';
                        textColor = '#2e7d32';
                    } else if (isSelected && !isCorrect) {
                        bgColor = '#ffebee';
                        borderColor = '#f44336';
                        textColor = '#c62828';
                    }
                } else if (isSelected) {
                    bgColor = '#e3f2fd';
                    borderColor = '#1976d2';
                    textColor = '#1565c0';
                }
                
                var cursor = showingFeedback ? 'default' : 'pointer';
                
                html += '<div class="mcq-option" data-id="' + opt.id + '" style="display: flex; align-items: center; gap: 16px; padding: 16px 20px; background: ' + bgColor + '; border: 2px solid ' + borderColor + '; border-radius: 8px; cursor: ' + cursor + '; transition: all 0.15s ease;">';
                html += '<div style="width: 32px; height: 32px; border: 2px solid ' + borderColor + '; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-weight: 700; color: ' + textColor + '; background: ' + (isSelected || (showingFeedback && isCorrect) ? borderColor : 'transparent') + '; color: ' + (isSelected || (showingFeedback && isCorrect) ? '#fff' : textColor) + ';">' + opt.id.toUpperCase() + '</div>';
                html += '<div style="flex: 1; font-size: 1em; color: ' + textColor + ';">' + opt.text + '</div>';
                if (showingFeedback && isCorrect) {
                    html += '<span style="color: #4caf50; font-size: 1.2em;">✓</span>';
                } else if (showingFeedback && isSelected && !isCorrect) {
                    html += '<span style="color: #f44336; font-size: 1.2em;">✗</span>';
                }
                html += '</div>';
            });
            
            html += '</div>';
            
            // Feedback
            if (showingFeedback) {
                var isCorrect = selectedAnswer === q.correct;
                html += '<div style="background: ' + (isCorrect ? '#e8f5e9' : '#fff3e0') + '; border: 2px solid ' + (isCorrect ? '#4caf50' : '#ff9800') + '; border-radius: 8px; padding: 16px 20px; margin-bottom: 20px;">';
                html += '<div style="font-weight: 600; color: ' + (isCorrect ? '#2e7d32' : '#e65100') + '; margin-bottom: 8px;">' + (isCorrect ? '✓ Correct!' : '✗ Not quite') + '</div>';
                html += '<div style="color: #333; font-size: 0.95em; line-height: 1.5;">' + q.explanation + '</div>';
                html += '</div>';
            }
            
            // Buttons
            html += '<div style="display: flex; gap: 12px; justify-content: flex-end;">';
            if (!showingFeedback) {
                html += '<button id="submitBtn" style="padding: 12px 28px; background: ' + (selectedAnswer ? '#4caf50' : '#ccc') + '; color: white; border: none; border-radius: 6px; cursor: ' + (selectedAnswer ? 'pointer' : 'not-allowed') + '; font-size: 1em; font-weight: 600;"' + (selectedAnswer ? '' : ' disabled') + '>Submit Answer</button>';
            } else {
                html += '<button id="nextQBtn" style="padding: 12px 28px; background: #1976d2; color: white; border: none; border-radius: 6px; cursor: pointer; font-size: 1em; font-weight: 600;">' + (currentIndex < questions.length - 1 ? 'Next Question' : 'See Results') + '</button>';
            }
            html += '</div>';
        }
        
        html += '</div>';
        
        root.innerHTML = html;
        attachEvents();
    }
    
    function attachEvents() {
        var options = document.querySelectorAll('.mcq-option');
        var submitBtn = document.getElementById('submitBtn');
        var nextQBtn = document.getElementById('nextQBtn');
        var retakeBtn = document.getElementById('retakeBtn');
        
        options.forEach(function(opt) {
            opt.addEventListener('click', function() {
                if (!showingFeedback) {
                    selectedAnswer = this.dataset.id;
                    render();
                }
            });
            
            if (!showingFeedback) {
                opt.addEventListener('mouseenter', function() {
                    if (selectedAnswer !== this.dataset.id) {
                        this.style.borderColor = '#1976d2';
                        this.style.background = '#f5f9ff';
                    }
                });
                opt.addEventListener('mouseleave', function() {
                    if (selectedAnswer !== this.dataset.id) {
                        this.style.borderColor = '#e0e0e0';
                        this.style.background = '#fff';
                    }
                });
            }
        });
        
        if (submitBtn) {
            submitBtn.addEventListener('click', function() {
                if (selectedAnswer) {
                    var q = questions[currentIndex];
                    if (selectedAnswer === q.correct) {
                        score++;
                    }
                    showingFeedback = true;
                    render();
                }
            });
        }
        
        if (nextQBtn) {
            nextQBtn.addEventListener('click', function() {
                currentIndex++;
                selectedAnswer = null;
                showingFeedback = false;
                render();
            });
        }
        
        if (retakeBtn) {
            retakeBtn.addEventListener('click', function() {
                init();
            });
        }
    }
    
    if (typeof NodeList.prototype.forEach !== 'function') {
        NodeList.prototype.forEach = Array.prototype.forEach;
    }
    
    init();
})();
</script>

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>